# Switch to a local model

In this notebook, we will:
1. Start a local model with Ollama.
2. Classify the same 10 reviews with the local model and the API model.
3. Compare accuracy, speed, cost and privacy.

**Data:** uses `data/processed/silver_labels.csv`, created once with [classify_reviews.ipynb](../03_Classify_Reviews/classify_reviews.ipynb). Reuse the file; no new data preparation is needed.


## Libraries and settings

In [1]:
# Libraries
import os
import subprocess
import time
import platform
from datetime import datetime
from pathlib import Path
from platform import python_version

import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import accuracy_score

# Folder with the data (relative to the folder of this notebook)
DATA = Path("../data")

# API model and course price assumptions in USD per 1 million tokens
MODEL = "gpt-6-luna"
PRICE_INPUT_PER_M = 0.10
PRICE_OUTPUT_PER_M = 0.50

# Local model: small enough for a codespace without a graphics card
LOCAL_MODEL = "qwen3:1.7b"

# The API key is a Codespaces secret; outside Codespaces it is read from .env
load_dotenv("../.env")

# The client sends our requests and retries when the rate limit is reached
client = OpenAI(max_retries=8)

# Show current working directory
print(os.getcwd())

/workspaces/llm_fundamentals/05_Local_Model


## Step 1: Start Ollama and download the model

The cell has three steps: install Ollama if it is missing (in the codespace it is already installed), start the Ollama server, and download the model (about 1.4 GB, only the first time). Lines that start with `!` are shell commands.

In [2]:
# 1. Install Ollama if it is missing (about 2 minutes)
!which ollama > /dev/null || curl -fsSL https://ollama.com/install.sh | sh

# 2. Start the Ollama server in the background.
# If the server is already running, this does nothing.
subprocess.Popen(
    ["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(3)

# 3. Download the model (only the first time) and list the installed models
!ollama pull {LOCAL_MODEL} 2>&1 | tail -n 1
!ollama list

success 


]11;?\

NAME          ID              SIZE      MODIFIED      
qwen3:1.7b    8f68893c685c    1.4 GB    5 seconds ago    


### Connect the client

Ollama listens on this computer at `localhost:11434`. We use the same OpenAI library as before with a local address (`base_url`) and model name. Ollama ignores the API key, but the library wants one.

In [3]:
# A client that talks to Ollama on this computer
local = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")


## Step 2: Classify 10 reviews with both models

We use 10 reviews (5 per chain) with silver labels from notebook 03. Both models get the same prompt.

In [4]:
# Reviews with silver labels from notebook 03
silver_path = DATA / "processed/silver_labels.csv"
silver = pd.read_csv(silver_path)

# 5 random reviews per chain, the same inputs for both models
test = silver.groupby("chain").sample(n=5, random_state=42)
test = test.reset_index(drop=True)

# The same classification prompt for both models
PROMPT = (
    "Classify the sentiment of this customer review. "
    "Answer with exactly one word: positive, negative or mixed.\n"
    "\n"
    "Review: "
)

test[["id", "chain", "stars", "silver", "text"]].head(5)

,id,chain,stars,silver,text
0,R009,IKEA,3.0,negative,I was excited about making a good review. Howe...
1,R185,IKEA,1.0,negative,I want to start by saying before this experien...
2,R152,IKEA,1.0,mixed,"Wow, the customer service here has fallen apar..."
3,R028,IKEA,3.0,positive,Good furniture. I like how cheap everything is...
4,R079,IKEA,1.0,negative,Well don't even try to order a new mattress fr...


### Local model

We clean each answer before we compare it with the silver label: remove spaces at both ends, use lower case and remove a final period. An answer that is then not exactly one of the three labels counts as an error.

In [5]:
# Local model: one review after the other, and we stop the time
labels = []
start = time.perf_counter()

for text in test["text"]:
    r = local.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[{"role": "user", "content": PROMPT + text}],
        reasoning_effort="none",
    )

    # Clean the answer: no spaces around it, lower case, no final period
    answer = r.choices[0].message.content
    answer = answer.strip().lower()
    labels.append(answer.rstrip("."))

local_seconds = (time.perf_counter() - start) / len(test)
test["local"] = labels

print("Seconds per review:", round(local_seconds, 2))

Seconds per review: 4.91


### API model

The same reviews go to the API model. `store=False` asks the provider not to keep the request. **Every run of this cell sends new requests and incurs API charges.** Their size depends on the texts and generated output tokens. The calculation below estimates them using the course prices.

In [6]:
# API model: the same reviews, and we add up the tokens
labels = []
input_tokens = 0
output_tokens = 0
start = time.perf_counter()

for text in test["text"]:
    r = client.responses.create(
        model=MODEL,
        input=PROMPT + text,
        reasoning={"effort": "low"},
        store=False,
    )

    # Clean the answer in the same way as for the local model
    answer = r.output_text.strip().lower()
    labels.append(answer.rstrip("."))

    input_tokens += r.usage.input_tokens
    output_tokens += r.usage.output_tokens

api_seconds = (time.perf_counter() - start) / len(test)
test["api"] = labels

# Cost: tokens / 1 million x price per 1 million tokens
api_cost = (
    input_tokens * PRICE_INPUT_PER_M + output_tokens * PRICE_OUTPUT_PER_M
) / 1_000_000

print("Seconds per review:", round(api_seconds, 2))
print("Input tokens:", input_tokens, "| output tokens:", output_tokens)

Seconds per review: 1.25
Input tokens: 1853 | output tokens: 211


## Step 3: Compare accuracy, speed and cost

In [7]:
# Accuracy against the silver labels, time per review and API cost
comparison = pd.DataFrame(
    {
        LOCAL_MODEL: [
            accuracy_score(test["silver"], test["local"]),
            local_seconds,
            0.0,
        ],
        MODEL: [
            accuracy_score(test["silver"], test["api"]),
            api_seconds,
            api_cost,
        ],
    },
    index=[
        "accuracy (silver labels)",
        "seconds per review",
        f"API cost in USD ({len(test)} reviews)",
    ],
)
comparison.round(4)

,qwen3:1.7b,gpt-6-luna
accuracy (silver labels),0.8000,0.9000
seconds per review,4.9149,1.2530
API cost in USD (10 reviews),0.0000,0.0003


In [8]:
# Reviews where at least one model disagrees with the silver label
pd.set_option("display.max_colwidth", 300)
errors = test[
    (test["local"] != test["silver"]) | (test["api"] != test["silver"])
]
errors[["id", "silver", "local", "api", "text"]]

,id,silver,local,api,text
2,R152,mixed,negative,negative,"Wow, the customer service here has fallen apart. I had 4 employee interactions and 3 were terrible. The first was with a person at customer service in the warehouse area. I asked him where their very common <PERSON> shelving was. He had never heard of it and pointed at a broken computer and sai..."
8,R347,positive,mixed,positive,"Clean efficient <PERSON>, newly renovated but in a rough part of downtown near the St Joseph Shrine. \n\nEasy in and out, can order at the automated kiosk and have dedicated staff to keep the dining room clean. \n\nAlso there is a 30 minute limit to how long you can be here, and no outside food ..."


## Interpret the comparison

- **Accuracy:** read the measured scores and inspect the disagreements below the table. Which model makes more errors in this sample? Do not assume the winner from the model size.
- **Fair comparison?** The silver labels were made by `gpt-6-luna` itself. A model agrees with its own labels more easily; a fair comparison needs gold labels made by people.
- **10 reviews are few:** one additional correct or incorrect prediction changes the accuracy by 10 percentage points. For a real decision you would test many more.
- **Cost:** the local model has no price per token, but it is not free: it needs hardware, electricity and somebody who runs it.
- **Privacy:** with a local model, the reviews never leave the computer. That can be the deciding argument.

## Check your understanding

Answer briefly using the outputs above. Try first, then open the hints.

1. What changes when the client points to localhost instead of the external API? Where are the review texts processed?
2. Use your comparison table: which model is more accurate and which is faster? Can these results establish that one model is always better?
3. Why can silver labels from the API model favour that model in the comparison? What would make the reference more independent?
4. Why do the measured seconds only describe this computer at this moment? Why do zero API costs not mean that a local model is free to operate?
5. Give one requirement favouring local processing and one favouring an API. Before choosing either, which simpler baseline could you test for this task?

**Your answers:**

1. …
2. …
3. …
4. …
5. …

<details>
<summary style="color: #B8860B; font-weight: bold; cursor: pointer;">Hints for self-check</summary>

1. The base URL and model change. Ollama serves the local model on this computer; the external API processes the submitted text at the provider.

2. Use the measured values, even if they differ from expectations. Ten reviews and one machine are weak evidence for a general claim.

3. The API model may reproduce its own reference labels more easily. Independently created human gold labels provide a fairer reference.

4. The time depends on the hardware, the network and the load at the moment of the run. Local operation still needs hardware, electricity and maintenance.

5. For example: a requirement to keep texts on the machine favours local processing; limited hardware or low measured latency may favour an API. Test TF-IDF with a classifier as a baseline.

</details>

## Optional: test a larger sample

If you have time, change `sample(n=5, ...)` to `sample(n=15, ...)` per chain (if enough valid silver labels are available) in step 2 and rerun the comparison. Are the same conclusions supported? This takes longer and sends new API requests. Other extensions are unnecessary for the core comparison.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [9]:
# Show operating system, date/time and Python version
print("-----------------------------------")
print(os.name.upper())
print(platform.system(), "|", platform.release())
print("Datetime:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print("Python Version:", python_version())
print("-----------------------------------")


-----------------------------------
POSIX
Linux | 6.8.0-1070-azure
Datetime: 2026-11-13 09:42:04
Python Version: 3.11.18
-----------------------------------
